# Part I — Fixed-state ADAPT-VQE gradient-measurement diagnostics

This notebook implements the Part I work order: three measurement strategies for the generator-selection step of gradient-based ADAPT-VQE, compared on the nine Part I benchmark cases.

**Problem.** Given a molecular Hamiltonian $H$, a current ADAPT-VQE state $|\psi\rangle$ and a UCCSD generator pool $\{G_i\}$, estimate the measurement cost needed to identify

$$i^\star = \arg\max_i |g_i|, \qquad g_i = \langle\psi|[H,G_i]|\psi\rangle .$$

**Method names** (used everywhere below, including file names):

| Code | Name | What is measured |
|---|---|---|
| `M1_FCUG` | M1: FC-UG, fully commuting universal groups | One global FC grouping of $\mathcal B_0=\bigcup_i \mathrm{supp}(C_i)$; the full gradient vector is estimated. |
| `M2_BAIFCIG` | M2: BAI-FC-IG, BAI with fully commuting individual gradients | Each $C_i$ is grouped on its own support; best-arm identification over the arms; no sharing between gradients. |
| `M3_BAIFCUG` | M3: BAI-FC-UG, BAI with fully commuting universal groups | The M1 parent groups, restricted at each BAI stage to the support of the surviving generators. |

**Scope.** All numbers are oracle planning estimates. The state is known exactly, and exact gradients and exact fragment variances are used to estimate shot requirements. There is no finite-shot sampling, no frozen core, no qubit tapering and no Part II coefficient splitting. The CISD state is a correlated proxy for a later ADAPT state.

In [1]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
import json, time
import numpy as np
import pandas as pd
import openfermion as of
import pyscf
print("openfermion", of.__version__, "| pyscf", pyscf.__version__, "| numpy", np.__version__)

openfermion 1.8.1 | pyscf 2.14.0 | numpy 2.4.4


## 1. Pauli strings and fully commuting groups

A Pauli string on $n$ qubits is stored as two bit masks $(x, z)$. Qubit 0 is the most significant bit, which is the ordering OpenFermion uses for state vectors. Two strings commute when the symplectic product $x_1\cdot z_2 + z_1\cdot x_2$ is even.

`group_fc` is a deterministic first-fit grouping. Paulis are visited in a given order, and each one joins the first existing group whose members it all commutes with; otherwise it opens a new group. No random numbers are involved.

In [2]:
import numpy as np

def popcount(a):
    return np.bitwise_count(np.asarray(a, dtype=np.uint64)).astype(np.int64)


def term_to_xz(term, nq):
    x = z = 0
    for q, p in term:
        bit = 1 << (nq - 1 - q)
        if p in "XY":
            x |= bit
        if p in "ZY":
            z |= bit
    return x, z


def xz_to_label(x, z, nq):
    s = []
    for q in range(nq):
        bit = 1 << (nq - 1 - q)
        s.append({(0, 0): "I", (1, 0): "X", (0, 1): "Z", (1, 1): "Y"}[(bool(x & bit), bool(z & bit))])
    return "".join(s)


def anticommutes(x1, z1, x2, z2):
    return (popcount((x1 & z2) ^ (z1 & x2)) & 1).astype(bool)


def group_fc(xs, zs, order):
    gid = -np.ones(len(xs), dtype=np.int64)
    mx, mz, mg = [], [], []
    ngroups = 0
    ax = np.zeros(len(xs), dtype=np.uint64); az = ax.copy(); ag = np.zeros(len(xs), dtype=np.int64)
    n = 0
    for l in order:
        if n:
            bad = anticommutes(ax[:n], az[:n], np.uint64(xs[l]), np.uint64(zs[l]))
            badg = np.bincount(ag[:n][bad], minlength=ngroups)
            ok = np.flatnonzero(badg == 0)
        else:
            ok = []
        g = ok[0] if len(ok) else ngroups
        if g == ngroups:
            ngroups += 1
        gid[l] = g
        ax[n] = xs[l]; az[n] = zs[l]; ag[n] = g; n += 1
    return gid, ngroups

## 2. Clifford diagonalization of a context

Part I only needs the groups, but each group is also given an explicit measurement circuit $U_\alpha$ so that its two-qubit gate count can be reported. The generators of the group are row-reduced over GF(2). Hadamards make the $X$ block full rank, CNOTs reduce it to the identity, S and CZ gates clear the $Z$ block, and final Hadamards turn every generator into a $Z$ string. `conjugate` tracks each Pauli through the circuit, including its sign.

In [3]:
import numpy as np


def gf2_basis(xs, zs, nq):
    rows = []
    piv = []
    for x, z in zip(xs, zs):
        v = (int(x) << nq) | int(z)
        for p, r in zip(piv, rows):
            if v >> p & 1:
                v ^= r
        if v:
            p = v.bit_length() - 1
            for k in range(len(rows)):
                if rows[k] >> p & 1:
                    rows[k] ^= v
            rows.append(v); piv.append(p)
    X = np.array([[(r >> (nq + nq - 1 - q)) & 1 for q in range(nq)] for r in rows], dtype=np.uint8).reshape(-1, nq)
    Z = np.array([[(r >> (nq - 1 - q)) & 1 for q in range(nq)] for r in rows], dtype=np.uint8).reshape(-1, nq)
    return X, Z


def diagonalizing_circuit(xs, zs, nq):
    X, Z = gf2_basis(xs, zs, nq)
    k = X.shape[0]
    gates = []

    def h(q):
        gates.append(("H", q)); X[:, q], Z[:, q] = Z[:, q].copy(), X[:, q].copy()

    def s(q):
        gates.append(("S", q)); Z[:, q] ^= X[:, q]

    def cx(c, t):
        gates.append(("CX", c, t)); X[:, t] ^= X[:, c]; Z[:, c] ^= Z[:, t]

    def cz(a, b):
        gates.append(("CZ", a, b)); Z[:, a] ^= X[:, b]; Z[:, b] ^= X[:, a]

    def rref():
        pivots = []
        r = 0
        for q in range(nq):
            rows = np.flatnonzero(X[r:, q]) + r
            if len(rows) == 0:
                continue
            p = rows[0]
            if p != r:
                X[[r, p]] = X[[p, r]]; Z[[r, p]] = Z[[p, r]]
            for rr in np.flatnonzero(X[:, q]):
                if rr != r:
                    X[rr] ^= X[r]; Z[rr] ^= Z[r]
            pivots.append(q); r += 1
            if r == k:
                break
        return pivots

    while True:
        pivots = rref()
        if len(pivots) == k:
            break
        r = len(pivots)
        free = [q for q in range(nq) if Z[r, q] and q not in pivots]
        h(free[0])
    for r, p in enumerate(pivots):
        for q in np.flatnonzero(X[r]):
            if q != p:
                cx(p, q)
    for r, p in enumerate(pivots):
        if Z[r, p]:
            s(p)
    for a in range(k):
        for b in range(a + 1, k):
            if Z[a, pivots[b]]:
                cz(pivots[a], pivots[b])
    for p in pivots:
        h(p)
    assert not X.any()
    return gates


def conjugate(x, z, gates, nq):
    xb = np.array([(x >> (nq - 1 - q)) & 1 for q in range(nq)], dtype=np.uint8)
    zb = np.array([(z >> (nq - 1 - q)) & 1 for q in range(nq)], dtype=np.uint8)
    r = 0
    for g in gates:
        if g[0] == "H":
            q = g[1]; r ^= xb[q] & zb[q]; xb[q], zb[q] = zb[q], xb[q]
        elif g[0] == "S":
            q = g[1]; r ^= xb[q] & zb[q]; zb[q] ^= xb[q]
        elif g[0] == "CX":
            c, t = g[1], g[2]
            r ^= xb[c] & zb[t] & (xb[t] ^ zb[c] ^ 1)
            xb[t] ^= xb[c]; zb[c] ^= zb[t]
        elif g[0] == "CZ":
            a, b = g[1], g[2]
            for c, t, pre in ((a, b, True),):
                r ^= xb[t] & zb[t]; xb[t], zb[t] = zb[t], xb[t]
                r ^= xb[c] & zb[t] & (xb[t] ^ zb[c] ^ 1)
                xb[t] ^= xb[c]; zb[c] ^= zb[t]
                r ^= xb[t] & zb[t]; xb[t], zb[t] = zb[t], xb[t]
    assert not xb.any()
    zm = 0
    for q in range(nq):
        if zb[q]:
            zm |= 1 << (nq - 1 - q)
    return int(r), zm


def apply_gates(psi, gates, nq):
    psi = psi.reshape([2] * nq).copy()
    s2 = 1 / np.sqrt(2)
    for g in gates:
        if g[0] == "H":
            q = g[1]
            a = np.take(psi, 0, axis=q); b = np.take(psi, 1, axis=q)
            psi = np.stack([(a + b) * s2, (a - b) * s2], axis=q)
        elif g[0] == "S":
            q = g[1]
            idx = [slice(None)] * nq; idx[q] = 1
            psi[tuple(idx)] *= 1j
        elif g[0] == "CX":
            c, t = g[1], g[2]
            idx = [slice(None)] * nq; idx[c] = 1
            sub = psi[tuple(idx)]
            tt = t if t < c else t - 1
            psi[tuple(idx)] = np.flip(sub, axis=tt)
        elif g[0] == "CZ":
            a, b = g[1], g[2]
            idx = [slice(None)] * nq; idx[a] = 1; idx[b] = 1
            psi[tuple(idx)] *= -1
    return psi.reshape(-1)


def pauli_expect(psi, x, z):
    b = np.arange(len(psi), dtype=np.uint64)
    ny = int(popcount(np.uint64(x & z)))
    sgn = 1 - 2 * (popcount(b & np.uint64(z)) & 1)
    val = np.vdot(psi[b ^ np.uint64(x)], sgn * psi) * (1j) ** ny
    return val.real

## 3. Molecular Hamiltonians, UCCSD pool and reference states

* **Integrals.** RHF/STO-3G with PySCF. Molecular-orbital integrals are converted to spin-orbital form (even index = α, odd = β) and mapped with Jordan–Wigner in OpenFermion.
* **Point-group symmetry.** Used for LiH and H₂O, where it only fixes the phases of degenerate orbitals. For square H₄ Part I used the RHF solution **without** point-group symmetry: this reproduces the Part I support size (1,276 Pauli products) and gradients exactly.
* **Pool.** Spin-orbital UCCSD pool conserving $N_e$ and $S_z$ (26 generators for H₄, 92 for LiH, 140 for H₂O). Each generator is anti-Hermitian, $G=T-T^\dagger$.
* **States.** The HF determinant, and the CISD state: the lowest singlet in the space of the HF determinant plus all single and double excitations, from exact diagonalization. The energy agrees with PySCF CISD.

In [4]:
import numpy as np, itertools, time
from pyscf import gto, scf, ao2mo, fci, ci, mcscf
import openfermion as of
from openfermion.chem.molecular_data import spinorb_from_spatial
from openfermion.ops.representations import get_active_space_integrals


def build_molecule(atom, basis="sto-3g", n_frozen=0, charge=0, spin=0, symmetry=True):
    mol = gto.M(atom=atom, basis=basis, charge=charge, spin=spin, unit="Angstrom", symmetry=symmetry, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-10
    mf.kernel()
    if not mf.converged:
        mf = mf.newton().run()
    C = mf.mo_coeff
    h1 = C.T @ mf.get_hcore() @ C
    norb = C.shape[1]
    eri = ao2mo.restore(1, ao2mo.kernel(mol, C), norb)
    two = np.asarray(eri.transpose(0, 2, 3, 1), order="C")
    core = mol.energy_nuc()
    nelec = mol.nelectron
    if n_frozen:
        occ = list(range(n_frozen))
        act = list(range(n_frozen, norb))
        ecore, h1, two = get_active_space_integrals(h1, two, occ, act)
        core += ecore
        nelec -= 2 * n_frozen
    one_so, two_so = spinorb_from_spatial(h1, two)
    iop = of.InteractionOperator(core, one_so, 0.5 * two_so)
    fop = of.get_fermion_operator(iop)
    H = of.jordan_wigner(fop)
    H.compress(1e-10)
    nq = 2 * h1.shape[0]
    if n_frozen:
        cas = mcscf.CASCI(mf, nq // 2, nelec)
        efci = cas.kernel()[0]
    else:
        efci = fci.FCI(mf).kernel()[0]
    myci = ci.CISD(mf)
    if n_frozen:
        myci.frozen = n_frozen
    ecisd = myci.kernel()[0] + mf.e_tot
    return dict(H=H, nq=nq, nelec=nelec, e_hf=mf.e_tot, e_fci=efci, e_cisd=ecisd, mol=mol)


def uccsd_pool(nq, nelec):
    occ = list(range(nelec)); vir = list(range(nelec, nq))
    pool, labels = [], []
    for i in occ:
        for a in vir:
            if i % 2 == a % 2:
                op = of.FermionOperator(((a, 1), (i, 0))) - of.FermionOperator(((i, 1), (a, 0)))
                pool.append(op); labels.append(f"S {i}->{a}")
    for i, j in itertools.combinations(occ, 2):
        for a, b in itertools.combinations(vir, 2):
            if (i % 2 + j % 2) == (a % 2 + b % 2):
                if {i % 2, j % 2} != {a % 2, b % 2} and (i % 2 + j % 2) == 1:
                    continue
                op = of.FermionOperator(((b, 1), (a, 1), (j, 0), (i, 0)))
                op = op - of.hermitian_conjugated(op)
                pool.append(op); labels.append(f"D {i},{j}->{a},{b}")
    qpool = []
    for op in pool:
        q = of.jordan_wigner(op); q.compress(1e-12); qpool.append(q)
    return qpool, labels


def sector_basis(nq, nelec):
    idx = []
    for b in range(2 ** nq):
        occ = [q for q in range(nq) if b >> (nq - 1 - q) & 1]
        if len(occ) == nelec and sum(1 for q in occ if q % 2 == 0) == nelec // 2:
            idx.append(b)
    return np.array(idx)


def hf_state(nq, nelec):
    psi = np.zeros(2 ** nq, dtype=complex)
    psi[sum(1 << (nq - 1 - q) for q in range(nelec))] = 1
    return psi


def excitation_level(b, nq, nelec):
    occ = {q for q in range(nq) if b >> (nq - 1 - q) & 1}
    return len(occ - set(range(nelec)))


def cisd_state(Hs, nq, nelec, spin_penalty=1.0):
    basis = sector_basis(nq, nelec)
    sub = np.array([b for b in basis if excitation_level(b, nq, nelec) <= 2])
    S2 = of.get_sparse_operator(of.jordan_wigner(of.s_squared_operator(nq // 2)), nq).tocsr()
    Hsub = Hs[sub][:, sub].toarray()
    e, v = np.linalg.eigh(Hsub + spin_penalty * S2[sub][:, sub].toarray().real)
    psi = np.zeros(2 ** nq, dtype=complex)
    psi[sub] = v[:, 0]
    return np.vdot(psi, Hs @ psi).real, psi


def fci_sector(Hs, nq, nelec):
    basis = sector_basis(nq, nelec)
    e, v = np.linalg.eigh(Hs[basis][:, basis].toarray())
    return e[0]

## 4. Shared Pauli coordinates and parent groups

For each generator the gradient operator $C_i=[H,G_i]$ is expanded in one shared Pauli coordinate system,

$$C_i=\sum_\ell A_{i\ell}R_\ell,\qquad \mu_\ell=\langle R_\ell\rangle_\psi,\qquad \boldsymbol g = A\boldsymbol\mu .$$

`Library` holds $A$ and the Pauli masks. `base_partition` builds the parent FC groups of $\mathcal B_0$ with the recorded convention: Paulis sorted by decreasing $\lVert A_{:,\ell}\rVert_2$, stable sort, then first-fit. `Contexts` attaches a diagonalization circuit to every group.

In [5]:
import numpy as np, time


class Library:
    def __init__(self, nq, A, xs, zs):
        self.nq = nq
        self.A_req = A
        self.K, self.L_req = A.shape
        self.xs = np.array(xs, dtype=np.uint64)
        self.zs = np.array(zs, dtype=np.uint64)
        self.rule = ["req"] * self.L_req

    @property
    def L(self):
        return len(self.xs)

    @property
    def A(self):
        out = np.zeros((self.K, self.L))
        out[:, : self.L_req] = self.A_req
        return out

    def __deepcopy__(self, memo):
        new = Library.__new__(Library)
        new.nq, new.A_req, new.K, new.L_req = self.nq, self.A_req, self.K, self.L_req
        new.xs, new.zs, new.rule = self.xs.copy(), self.zs.copy(), list(self.rule)
        return new

    def add_aux(self, x, z, rule):
        self.xs = np.append(self.xs, np.uint64(x))
        self.zs = np.append(self.zs, np.uint64(z))
        self.rule.append(rule)
        return self.L - 1

    def label(self, l):
        return xz_to_label(int(self.xs[l]), int(self.zs[l]), self.nq)


class Contexts:
    def __init__(self, lib, members):
        self.lib = lib
        self.members = [np.array(m, dtype=np.int64) for m in members]
        self.build_circuits()

    def build_circuits(self):
        lib = self.lib
        self.gates, self.sign, self.zimg, self.n2q = [], [], [], []
        for mem in self.members:
            g = diagonalizing_circuit(lib.xs[mem], lib.zs[mem], lib.nq)
            rz = [conjugate(int(lib.xs[l]), int(lib.zs[l]), g, lib.nq) for l in mem]
            self.gates.append(g)
            self.sign.append(np.array([1 - 2 * r for r, _ in rz], dtype=np.int8))
            self.zimg.append(np.array([z for _, z in rz], dtype=np.uint64))
            self.n2q.append(sum(1 for q in g if q[0] in ("CX", "CZ")))
        self.n2q = np.array(self.n2q)

    def __len__(self):
        return len(self.members)

    def membership(self):
        cnt = np.zeros(self.lib.L, dtype=np.int64)
        for m in self.members:
            cnt[m] += 1
        return cnt


def base_partition(lib):
    w = np.linalg.norm(lib.A_req, axis=0)
    gid, ng = group_fc(lib.xs[: lib.L_req], lib.zs[: lib.L_req], np.argsort(-w))
    return [np.flatnonzero(gid == g) for g in range(ng)]

## 5. Oracle shot proxy and the three algorithms

**Fragments.** Group $\alpha$ carries the fragment $F_{i\alpha}=\sum_{\ell\in\alpha}A_{i\ell}R_\ell$ of gradient $i$, with oracle variance

$$\sigma^2_{i\alpha}=\langle F_{i\alpha}^2\rangle_\psi-\langle F_{i\alpha}\rangle_\psi^2 .$$

It is computed exactly as $\lVert F_{i\alpha}\psi\rVert^2-\langle\psi|F_{i\alpha}\psi\rangle^2$ from the state vector.

**Confidence factor.** $z=\Phi^{-1}(1-\delta/2N)$ with $\delta=0.05$ and $N$ the pool size (family-wise Bonferroni over the pool). A radius $r$ corresponds to a one-standard-error target $\epsilon=r/z$.

**Gaps.** Sort $|g|$ in decreasing order. The winner $w$ has the largest $|g_i|$, and the gap is $|g_{(1)}|-|g_{(2)}|$. Arm $i\neq w$ has $\Delta_i=|g_w|-|g_i|$; the winner is given $\Delta_w=\text{gap}$.

**Allocation with shared groups.** When several gradients are read from the same groups, the shots $n_\alpha$ are chosen to solve

$$\min_{n}\sum_\alpha n_\alpha\quad\text{s.t.}\quad \sum_\alpha \sigma^2_{i\alpha}/n_\alpha\le\epsilon_i^2\ \ \forall i\in\mathcal A .$$

This is solved by multiplicative dual ascent, $n_\alpha=\sqrt{\sum_i\lambda_i\sigma^2_{i\alpha}}$ and $\lambda_i\leftarrow\lambda_i(\mathrm{Var}_i/\epsilon_i^2)^{1/2}$, followed by a final rescaling so that every constraint holds. The reported cost is always that of a feasible allocation.

**M1: FC-UG.** All arms are resolved to the radius $r=\text{gap}/2$ with the shared parent groups. This is one allocation over all arms.

**M2: BAI-FC-IG.** Each arm is grouped on its own support. An arm is eliminated once its interval and the winner's interval separate, which with a common radius happens at $r_i=\Delta_i/2$. Eq. (16) of the Part I report gives the cost of arm $i$,

$$M_i=\Big(\sum_\alpha\sigma_{i\alpha}\Big)^2\Big/\epsilon_i^2,\qquad \epsilon_i=\Delta_i/(2z),$$

and M2 is the sum over arms.

**M3: BAI-FC-UG.** The parent groups of M1 are used, and arms are eliminated in order of decreasing $\Delta_i$. At each distinct value $d$ of $\Delta$, the active set is $\{i:\Delta_i\le d\}$. Its fragments, which are the parent groups restricted to the active support, are allocated to the common radius $d/2$. Shots are cumulative, so each context is charged the largest request it receives over the stages: $M3=\sum_\alpha\max_k n^{(k)}_\alpha$.

**M3 with support-triggered regrouping (M3-R).** Same stages as M3, plus one oracle-free rule. Before each stage, count the Pauli terms that appear in the supports of two or more surviving generators. Once that count is zero, the survivors no longer share any measured quantity, so the parent universal groups carry no sharing benefit for them. The surviving support $\mathcal B_r$ is then regrouped into fresh FC groups (sorted insertion ordered by $\lVert A_{\mathcal A_r,\ell}\rVert_2$), and the remaining stages use these groups. The shots already spent on the parent contexts are not thrown away. For each surviving arm, they give an independent estimate with variance $V^{\text{old}}_i=\sum_\alpha\sigma^2_{i\alpha}/n_\alpha$, and the new groups only need to supply the rest: $1/V^{\text{new}}_i\ge 1/\epsilon_i^2-1/V^{\text{old}}_i$. M3-R is the parent-context charge before the regroup plus the fresh-context charge after it.

The trigger uses only the Pauli supports of the generators that BAI itself has kept. It does not use which arm is winning, any gradient value or any variance, so it can be run without an oracle. Ordering the parent insertion by the leading gradients would also give small groups for the winner, but it assumes the answer the selection is paying to find.

In [6]:
import numpy as np
from scipy.stats import norm


def pauli_apply(psi, x, z):
    b = np.arange(len(psi), dtype=np.uint64)
    ny = int(popcount(np.uint64(x & z)))
    sgn = 1 - 2 * (popcount(b & np.uint64(z)) & 1)
    out = np.empty_like(psi)
    out[b ^ np.uint64(x)] = sgn * psi * (1j) ** ny
    return out


def fragment_variances(psi, xs, zs, A, groups):
    K = A.shape[0]
    q = np.zeros((K, len(groups)))
    mu = np.zeros((K, len(groups)))
    for a, mem in enumerate(groups):
        rows = np.flatnonzero(np.abs(A[:, mem]).max(1) > 0)
        if len(rows) == 0:
            continue
        P = np.array([pauli_apply(psi, int(xs[l]), int(zs[l])) for l in mem])
        F = A[np.ix_(rows, mem)] @ P
        m = (F @ psi.conj()).real
        q[rows, a] = np.maximum(np.einsum("ij,ij->i", F.conj(), F).real - m ** 2, 0.0)
        mu[rows, a] = m
    return q, mu


def z_value(n_arms, delta=0.05):
    return norm.ppf(1 - delta / (2 * n_arms))


def allocate(q, eps, iters=400, tol=1e-9):
    q = np.asarray(q, float)
    need = q.sum(0) > 0
    q = q[:, need]
    eps2 = np.asarray(eps, float) ** 2
    lam = np.ones(q.shape[0])
    best = None
    for it in range(iters):
        n = np.sqrt(np.maximum(lam @ q, 1e-300))
        V = (q / n).sum(1)
        scale = np.max(V / eps2)
        cost = scale * n.sum()
        if best is None or cost < best[0] - tol * best[0]:
            best = (cost, n * scale)
        lam = lam * np.sqrt(V / eps2)
        lam /= lam.max()
    out = np.zeros(len(need))
    out[need] = best[1]
    return best[0], out


def gaps(g):
    a = np.abs(g)
    w = int(np.argmax(a))
    s = np.sort(a)[::-1]
    gap = s[0] - s[1]
    D = a[w] - a
    D[w] = gap
    return w, gap, D


def m1_cost(q, g, z):
    w, gap, D = gaps(g)
    eps = np.full(q.shape[0], gap / 2 / z)
    active = q.sum(1) > 0
    cost, n = allocate(q[active], eps[active])
    return cost, n


def m2_cost(q_list, g, z):
    w, gap, D = gaps(g)
    per = np.zeros(len(g))
    for i, qi in enumerate(q_list):
        s = np.sqrt(qi).sum()
        if s == 0:
            continue
        eps = D[i] / 2 / z
        per[i] = s ** 2 / eps ** 2
    return per.sum(), per


def m3_cost(q, g, z):
    w, gap, D = gaps(g)
    levels = np.unique(np.round(D, 12))[::-1]
    charged = np.zeros(q.shape[1])
    stages = []
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (q.sum(1) > 0))
        if len(act) == 0:
            continue
        eps = np.full(len(act), d / 2 / z)
        cost, n = allocate(q[act], eps)
        charged = np.maximum(charged, n)
        stages.append(dict(radius=d / 2, active=len(act), stage_cost=cost))
    return charged.sum(), charged, stages


def per_gradient_groups(A, xs, zs):
    out = []
    for i in range(A.shape[0]):
        sup = np.flatnonzero(A[i])
        order = np.argsort(-np.abs(A[i, sup]))
        gid, ng = group_fc(xs[sup], zs[sup], order)
        out.append([sup[gid == k] for k in range(ng)])
    return out


def shared_fraction(A, active):
    cnt = (np.abs(A[active]) > 0).sum(0)
    union = int((cnt > 0).sum())
    return (cnt >= 2).sum() / max(union, 1), union


def regroup(A, xs, zs, active):
    sup = np.flatnonzero(np.abs(A[active]).max(0) > 0)
    order = np.argsort(-np.linalg.norm(A[np.ix_(active, sup)], axis=0), kind="stable")
    gid, ng = group_fc(xs[sup], zs[sup], order)
    return [sup[gid == k] for k in range(ng)]


def m3r_cost(psi, xs, zs, A, q, g, z, threshold=0.0):
    w, gap, D = gaps(g)
    levels = np.unique(np.round(D, 12))[::-1]
    parent = np.zeros(q.shape[1])
    fresh, qn, groups_new, trigger, v_old = None, None, None, None, None
    stages = []
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (q.sum(1) > 0))
        if len(act) == 0:
            continue
        frac, union = shared_fraction(A, act)
        if groups_new is None and frac <= threshold:
            groups_new = regroup(A, xs, zs, act)
            qn, _ = fragment_variances(psi, xs, zs, A[act], groups_new)
            rows = {i: r for r, i in enumerate(act)}
            fresh = np.zeros(len(groups_new))
            v_old = {}
            for i in act:
                used = q[i] > 0
                v_old[i] = np.sum(q[i][used] / parent[used]) if np.all(parent[used] > 0) else np.inf
            trigger = dict(radius=d / 2, active=len(act), union_terms=union, new_groups=len(groups_new))
        eps = np.full(len(act), d / 2 / z)
        if groups_new is None:
            cost, n = allocate(q[act], eps)
            parent = np.maximum(parent, n)
        else:
            need = []
            for i, e in zip(act, eps):
                inv = 1.0 / e ** 2 - (1.0 / v_old[i] if np.isfinite(v_old[i]) else 0.0)
                need.append(1.0 / inv if inv > 0 else np.inf)
            need = np.array(need)
            keep = np.isfinite(need)
            if keep.any():
                cost, n = allocate(qn[[rows[i] for i in act[keep]]], np.sqrt(need[keep]))
                fresh = np.maximum(fresh, n)
            else:
                cost = 0.0
        stages.append(dict(radius=d / 2, active=len(act), shared_fraction=float(frac), regrouped=groups_new is not None, stage_cost=cost))
    total = parent.sum() + (fresh.sum() if fresh is not None else 0.0)
    return total, stages, trigger, groups_new


def fragmentation(A, groups, i):
    gid = np.empty(A.shape[1], dtype=np.int64)
    for a, mem in enumerate(groups):
        gid[mem] = a
    return len(set(gid[np.flatnonzero(A[i])]))

## 6. Benchmark cases

The nine Part I cases. H₄ is a square with side 1.0 or 2.0 Å. H₂O uses the bond angle 104.52°, which reproduces the Part I gradients to six digits. `PART_I_REFERENCE` holds the values printed in the Part I report (Tables 2–4), which are compared with below. The H₂O M1 and M3 entries are empty because Part I had not computed them.

In [7]:
import numpy as np


def h4_square(a):
    return f"H 0 0 0; H {a} 0 0; H 0 {a} 0; H {a} {a} 0"


def water(r, angle=104.52):
    t = np.deg2rad(angle)
    return f"O 0 0 0; H {r} 0 0; H {r*np.cos(t):.6f} {r*np.sin(t):.6f} 0"


MOLECULES = {
    "LiH_3.0": dict(atom="Li 0 0 0; H 0 0 3.0", symmetry=True, label="LiH, R=3.0 A"),
    "H4_1.0": dict(atom=h4_square(1.0), symmetry=False, label="H4 square, side=1.0 A"),
    "H4_2.0": dict(atom=h4_square(2.0), symmetry=False, label="H4 square, side=2.0 A"),
    "H2O_eq": dict(atom=water(0.9572), symmetry=True, label="H2O eq., R(OH)=0.9572 A"),
    "H2O_1.75": dict(atom=water(1.75), symmetry=True, label="H2O stretch, R(OH)=1.75 A"),
}

CASES = [("LiH_3.0", "HF"), ("H4_1.0", "HF"), ("H4_1.0", "CISD"), ("H4_2.0", "HF"), ("H4_2.0", "CISD"),
         ("H2O_eq", "HF"), ("H2O_eq", "CISD"), ("H2O_1.75", "HF"), ("H2O_1.75", "CISD")]

PART_I_REFERENCE = {
    ("LiH_3.0", "HF"): dict(M1=2088380, M2=7368553, M3=1267508, contexts=980, terms=30196, g1=0.243187, g2=0.146799),
    ("H4_1.0", "HF"): dict(M1=182282619, M2=451756, M3=45857, contexts=356, terms=1276, g1=0.295316, g2=0.273776),
    ("H4_1.0", "CISD"): dict(M1=6066911, M2=53265, M3=9582, contexts=356, terms=1276, g1=0.142467, g2=0.024386),
    ("H4_2.0", "HF"): dict(M1=70335943, M2=112099, M3=9936, contexts=356, terms=1276, g1=0.315036, g2=0.298387),
    ("H4_2.0", "CISD"): dict(M1=7295352, M2=19188, M3=3441, contexts=356, terms=1276, g1=0.098961, g2=0.047102),
    ("H2O_eq", "HF"): dict(M1=None, M2=16545899, M3=None, contexts=9874, terms=81566, g1=0.304910, g2=0.203820),
    ("H2O_eq", "CISD"): dict(M1=None, M2=30149151647, M3=None, contexts=9874, terms=81566, g1=0.006925, g2=0.004861),
    ("H2O_1.75", "HF"): dict(M1=None, M2=9124527, M3=None, contexts=9874, terms=81566, g1=0.388493, g2=0.366638),
    ("H2O_1.75", "CISD"): dict(M1=None, M2=92693712, M3=None, contexts=9874, terms=81566, g1=0.106873, g2=0.089917),
}

## 7. Running the three methods on all nine cases

For each molecule the cell below does the following:
1. Builds $A$, the parent groups (M1/M3) and the per-gradient groups (M2).
2. For each requested state, computes the exact gradients and checks that the fragment means add up to them.
3. Evaluates M1, M2 and M3.
4. Writes every object Part I asks to keep, into files named with the method codes:
   * `*_A.npz`: coefficient matrix, Pauli labels, generator labels.
   * `*_M1_FCUG_M3_BAIFCUG_groups.json` and `*_M2_BAIFCIG_groups.json`: the groups and the grouping convention.
   * `*_M1_FCUG.npz`, `*_M2_BAIFCIG.npz`, `*_M3_BAIFCUG.npz`: fragment variances, shots per group, totals and $z$.
   * `*_M3_BAIFCUG_stages.json`: radius, active-set size and cost of every M3 stage.
   * `*_exact_gradients.json`: oracle gradients.

The full 14-qubit H₂O takes about 5–6 minutes per geometry on two cores.

In [8]:
OUT = "partI_results"
os.makedirs(OUT, exist_ok=True)
rows, grad_rows = [], []
ONLY = None
for mol_key, spec in MOLECULES.items():
    if ONLY and mol_key not in ONLY.split(","):
        continue
    t0 = time.time()
    m = build_molecule(spec["atom"], symmetry=spec["symmetry"])
    nq, ne = m["nq"], m["nelec"]
    pool, labels = uccsd_pool(nq, ne)
    Ds = []
    for G in pool:
        D = m["H"] * G - G * m["H"]
        D.compress(1e-10)
        Ds.append(D)
    terms = sorted(set().union(*[set(D.terms) for D in Ds]))
    pos = {t: i for i, t in enumerate(terms)}
    A = np.zeros((len(Ds), len(terms)))
    for i, D in enumerate(Ds):
        for t, c in D.terms.items():
            A[i, pos[t]] = c.real
    xz = np.array([term_to_xz(t, nq) for t in terms])
    lib = Library(nq, A, xz[:, 0], xz[:, 1])
    groups = base_partition(lib)
    ctx = Contexts(lib, groups)
    pg = per_gradient_groups(A, lib.xs, lib.zs)
    Hs = of.get_sparse_operator(m["H"], nq).tocsr()
    Gs = [of.get_sparse_operator(G, nq).tocsr() for G in pool]
    e_fci = fci_sector(Hs, nq, ne)
    states = {"HF": hf_state(nq, ne)}
    e_cisd, states["CISD"] = cisd_state(Hs, nq, ne)
    np.savez_compressed(f"{OUT}/{mol_key}_A.npz", A=A, paulis=np.array([lib.label(l) for l in range(lib.L)]), generators=np.array(labels))
    json.dump(dict(convention="sorted insertion: Paulis ordered by decreasing ||A[:,l]||_2 (stable argsort on the term list sorted lexicographically by OpenFermion term tuple), first-fit into the first fully commuting group; no randomness",
                   groups=[[lib.label(l) for l in grp] for grp in groups], n_2q=[int(x) for x in ctx.n2q]),
              open(f"{OUT}/{mol_key}_M1_FCUG_M3_BAIFCUG_groups.json", "w"))
    json.dump(dict(convention="per-gradient sorted insertion on supp(C_i), ordered by decreasing |A[i,l]|",
                   groups={labels[i]: [[lib.label(l) for l in grp] for grp in pg[i]] for i in range(len(pg))}),
              open(f"{OUT}/{mol_key}_M2_BAIFCIG_groups.json", "w"))
    for st in ("HF", "CISD"):
        if (mol_key, st) not in CASES:
            continue
        psi = states[st]
        hp = Hs @ psi
        g = np.array([2 * np.vdot(hp, G @ psi).real for G in Gs])
        z = z_value(len(g))
        q, mu = fragment_variances(psi, lib.xs, lib.zs, A, groups)
        err = np.abs(mu.sum(1) - g).max()
        print(mol_key, st, "max |sum of fragment means - g| =", err, flush=True)
        assert err < 1e-7
        ql = [fragment_variances(psi, lib.xs, lib.zs, A[i:i + 1], pg[i])[0][0] for i in range(len(g))]
        c1, n1 = m1_cost(q, g, z)
        c2, per2 = m2_cost(ql, g, z)
        c3, n3, stages = m3_cost(q, g, z)
        c3r, stages_r, trig, groups_r = m3r_cost(psi, lib.xs, lib.zs, A, q, g, z)
        top2 = np.argsort(-np.abs(g))[:2]
        frag = [fragmentation(A, groups, i) for i in top2]
        own = [len(pg[i]) for i in top2]
        w, gap, D = gaps(g)
        order = np.argsort(-np.abs(g))
        tag = f"{mol_key}_{st}"
        np.savez_compressed(f"{OUT}/{tag}_M1_FCUG.npz", fragment_variances=q, shots_per_context=n1, total=c1, z=z)
        np.savez_compressed(f"{OUT}/{tag}_M2_BAIFCIG.npz", total=c2, per_arm=per2, z=z,
                            **{f"fragment_variances_{i}": ql[i] for i in range(len(ql))})
        np.savez_compressed(f"{OUT}/{tag}_M3_BAIFCUG.npz", fragment_variances=q, shots_per_context=n3, total=c3, z=z)
        json.dump(stages, open(f"{OUT}/{tag}_M3_BAIFCUG_stages.json", "w"))
        json.dump(dict(rule="regroup the surviving support once the surviving generators share no Pauli term",
                       trigger=trig, stages=stages_r,
                       regrouped_contexts=[[lib.label(l) for l in grp] for grp in groups_r] if groups_r else None),
                  open(f"{OUT}/{tag}_M3R_BAIFCUG_regroup.json", "w"), default=float)
        rows.append(dict(case=mol_key, state=st, label=spec["label"], qubits=nq, pool=len(pool), universal_terms=lib.L,
                         contexts=len(groups), nonzero_g=int(np.sum(np.abs(g) > 1e-8)), top=labels[order[0]],
                         g1=abs(g[order[0]]), g2=abs(g[order[1]]), gap=gap, z=z, M1=c1, M2=c2, M3=c3, M3R=c3r,
                         regroup_active=trig["active"] if trig else 0, regroup_radius=trig["radius"] if trig else np.nan,
                         regroup_contexts=trig["new_groups"] if trig else 0,
                         winner_parent_contexts=frag[0], winner_own_groups=own[0],
                         runner_parent_contexts=frag[1], runner_own_groups=own[1],
                         M2_groups=int(sum(len(x) for x in pg)), mean_2q=float(ctx.n2q.mean()),
                         E_state_minus_FCI=float(np.vdot(psi, Hs @ psi).real - e_fci)))
        json.dump(dict(gradients=g.tolist(), labels=labels), open(f"{OUT}/{tag}_exact_gradients.json", "w"))
        print(rows[-1]["case"], st, f"M1={c1:.4g} M2={c2:.4g} M3={c3:.4g} M3R={c3r:.4g}", trig, flush=True)
    print(mol_key, "done", round(time.time() - t0), "s", flush=True)
prev = f"{OUT}/partI_summary.csv"
df = pd.DataFrame(rows)
if ONLY and os.path.exists(prev):
    old = pd.read_csv(prev)
    df = pd.concat([old[~old.case.isin(df.case)], df], ignore_index=True)
df.to_csv(prev, index=False)

LiH_3.0 HF max |sum of fragment means - g| = 4.163336342344337e-17


LiH_3.0 HF M1=2.358e+06 M2=9.09e+05 M3=4.162e+05 M3R=4.162e+05 None


LiH_3.0 done 34 s


H4_1.0 HF max |sum of fragment means - g| = 1.3877787807814457e-17


H4_1.0 HF M1=2.163e+06 M2=9.641e+05 M3=3.557e+05 M3R=3.43e+05 {'radius': np.float64(0.010769688154), 'active': 2, 'union_terms': 448, 'new_groups': 9}


...

H2O_1.75 HF max |sum of fragment means - g| = 2.220446049250313e-16


H2O_1.75 HF M1=2.19e+09 M2=1.072e+07 M3=1.142e+07 M3R=8.449e+06 {'radius': np.float64(0.0109278437995), 'active': 2, 'union_terms': 4736, 'new_groups': 150}


H2O_1.75 CISD max |sum of fragment means - g| = 2.213507155346406e-14


H2O_1.75 CISD M1=3.002e+09 M2=1.082e+08 M3=8.49e+07 M3R=8.184e+07 {'radius': np.float64(0.0084782529225), 'active': 2, 'union_terms': 4736, 'new_groups': 150}


H2O_1.75 done 318 s


## 8. Results

### Table 2 — Observable structure
Universal support, number of parent FC groups and number of nonzero gradients ($|g_i|>10^{-8}$). The Part I values are shown next to ours. Group counts depend on the grouping routine and ordering and are comparable only within one routine, as the Part I report notes.

In [9]:
summary = pd.read_csv(f"{OUT}/partI_summary.csv")
key = {(r.case, r.state): i for i, r in summary.iterrows()}
summary = summary.loc[[key[c] for c in CASES]].reset_index(drop=True)
ref = pd.DataFrame([dict(case=c, state=s, **PART_I_REFERENCE[(c, s)]) for c, s in CASES])
t2 = pd.DataFrame(dict(Case=summary.label, State=summary.state, Qubits=summary.qubits, Pool=summary.pool,
                       Universal_terms=summary.universal_terms, Part_I_terms=ref.terms,
                       FC_UG_contexts=summary.contexts, Part_I_contexts=ref.contexts, Nonzero_g=summary.nonzero_g,
                       mean_2q_gates_per_context=summary.mean_2q.round(1)))
t2

,Case,State,Qubits,Pool,Universal_terms,Part_I_terms,FC_UG_contexts,Part_I_contexts,Nonzero_g,mean_2q_gates_per_context
0,"LiH, R=3.0 A",HF,12,92,30196,30196,956,980,30,24.6
1,"H4 square, side=1.0 A",HF,8,26,1276,1276,37,356,10,7.2
2,"H4 square, side=1.0 A",CISD,8,26,1276,1276,37,356,10,7.2
3,"H4 square, side=2.0 A",HF,8,26,1276,1276,33,356,10,6.2
4,"H4 square, side=2.0 A",CISD,8,26,1276,1276,33,356,10,6.2
5,"H2O eq., R(OH)=0.9572 A",HF,14,140,81634,81566,2409,9874,48,34.9
6,"H2O eq., R(OH)=0.9572 A",CISD,14,140,81634,81566,2409,9874,58,34.9
7,"H2O stretch, R(OH)=1.75 A",HF,14,140,81594,81566,2431,9874,56,34.5
8,"H2O stretch, R(OH)=1.75 A",CISD,14,140,81594,81566,2431,9874,70,34.5


### Table 3 — Top-gradient gaps and M2: BAI-FC-IG
Smaller gaps make winner identification harder. The gradients are compared with the Part I values.

In [10]:
t3 = pd.DataFrame(dict(Case=summary.label, State=summary.state, Top_generator=summary.top,
                       g1=summary.g1.round(6), g2=summary.g2.round(6), Gap=summary.gap.round(6),
                       Part_I_g1=ref.g1, Part_I_g2=ref.g2,
                       M2_shots=summary.M2.round(0).astype(np.int64), Part_I_M2=ref.M2))
print("largest |g1 - Part I|:", np.abs(t3.g1 - t3.Part_I_g1).max(), " largest |g2 - Part I|:", np.abs(t3.g2 - t3.Part_I_g2).max())
t3

largest |g1 - Part I|: 0.0  largest |g2 - Part I|: 0.0


,Case,State,Top_generator,g1,g2,Gap,Part_I_g1,Part_I_g2,M2_shots,Part_I_M2
0,"LiH, R=3.0 A",HF,"D 2,3->10,11",0.243187,0.146799,0.096388,0.243187,0.146799,908972,7368553
1,"H4 square, side=1.0 A",HF,"D 2,3->6,7",0.295316,0.273776,0.021539,0.295316,0.273776,964052,451756
2,"H4 square, side=1.0 A",CISD,"D 0,1->6,7",0.142467,0.024386,0.118081,0.142467,0.024386,102363,53265
3,"H4 square, side=2.0 A",HF,"D 2,3->6,7",0.315036,0.298387,0.016649,0.315036,0.298387,456645,112099
4,"H4 square, side=2.0 A",CISD,"D 0,1->6,7",0.098961,0.047102,0.051859,0.098961,0.047102,90471,19188
5,"H2O eq., R(OH)=0.9572 A",HF,"D 4,5->12,13",0.304910,0.203820,0.101090,0.304910,0.203820,19510231,16545899
6,"H2O eq., R(OH)=0.9572 A",CISD,"D 4,5->12,13",0.006925,0.004861,0.002064,0.006925,0.004861,35812518352,30149151647
7,"H2O stretch, R(OH)=1.75 A",HF,"D 6,7->12,13",0.388493,0.366638,0.021856,0.388493,0.366638,10719210,9124527
8,"H2O stretch, R(OH)=1.75 A",CISD,"D 6,7->12,13",0.106873,0.089917,0.016957,0.106873,0.089917,108186576,92693712


### Table 4 — Completed three-method comparison
All nine rows, including H₂O, with oracle context-shots. $M1/M2>1$ means M2 is cheaper than M1, and $M2/M3>1$ means M3 is cheaper than M2.

In [11]:
t4 = pd.DataFrame(dict(Case=summary.label, State=summary.state,
                       M1_FCUG=summary.M1.round(0).astype(np.int64), M2_BAIFCIG=summary.M2.round(0).astype(np.int64),
                       M3_BAIFCUG=summary.M3.round(0).astype(np.int64),
                       M1_over_M2=(summary.M1 / summary.M2).round(2), M2_over_M3=(summary.M2 / summary.M3).round(2),
                       M1_over_M3=(summary.M1 / summary.M3).round(2)))
t4.to_csv(f"{OUT}/table4_three_methods.csv", index=False)
t4

,Case,State,M1_FCUG,M2_BAIFCIG,M3_BAIFCUG,M1_over_M2,M2_over_M3,M1_over_M3
0,"LiH, R=3.0 A",HF,2358286,908972,416237,2.59,2.18,5.67
1,"H4 square, side=1.0 A",HF,2163179,964052,355668,2.24,2.71,6.08
2,"H4 square, side=1.0 A",CISD,53255,102363,44972,0.52,2.28,1.18
3,"H4 square, side=2.0 A",HF,575592,456645,141420,1.26,3.23,4.07
4,"H4 square, side=2.0 A",CISD,43991,90471,29412,0.49,3.08,1.50
5,"H2O eq., R(OH)=0.9572 A",HF,106461625,19510231,14034153,5.46,1.39,7.59
6,"H2O eq., R(OH)=0.9572 A",CISD,250962481286,35812518352,25208640396,7.01,1.42,9.96
7,"H2O stretch, R(OH)=1.75 A",HF,2190235853,10719210,11421199,204.33,0.94,191.77
8,"H2O stretch, R(OH)=1.75 A",CISD,3001532061,108186576,84904189,27.74,1.27,35.35


### Comparison with the Part I preliminary numbers
Part I marked its M2 column as a planning diagnostic to be regenerated from saved fragments, and it had no M1/M3 values for H₂O. The ratio columns show how far the recomputed numbers are from the printed ones.

In [12]:
cmp_ = pd.DataFrame(dict(Case=summary.label, State=summary.state,
                         M1_ratio_new_over_PartI=summary.M1 / ref.M1.astype(float),
                         M2_ratio_new_over_PartI=summary.M2 / ref.M2.astype(float),
                         M3_ratio_new_over_PartI=summary.M3 / ref.M3.astype(float))).round(3)
cmp_

,Case,State,M1_ratio_new_over_PartI,M2_ratio_new_over_PartI,M3_ratio_new_over_PartI
0,"LiH, R=3.0 A",HF,1.129,0.123,0.328
1,"H4 square, side=1.0 A",HF,0.012,2.134,7.756
2,"H4 square, side=1.0 A",CISD,0.009,1.922,4.693
3,"H4 square, side=2.0 A",HF,0.008,4.074,14.233
4,"H4 square, side=2.0 A",CISD,0.006,4.715,8.548
5,"H2O eq., R(OH)=0.9572 A",HF,NaN,1.179,NaN
6,"H2O eq., R(OH)=0.9572 A",CISD,NaN,1.188,NaN
7,"H2O stretch, R(OH)=1.75 A",HF,NaN,1.175,NaN
8,"H2O stretch, R(OH)=1.75 A",CISD,NaN,1.167,NaN


### Why M3 lost to M2 on H₂O stretch (HF), and the regrouping fix
In Table 4 the only row with $M2/M3<1$ is H₂O at 1.75 Å (HF). The diagnostics below explain it. At the final stage only the two leading generators survive, and their supports share **no** Pauli term. The winner's 1,888 terms are spread over 680 parent contexts, against 58 groups when its commutator is grouped on its own (M2). Its measured fragments are therefore small pieces of large universal groups, and the sum of fragment standard deviations, which sets the Eq. (16) cost, is larger than in M2. The two leading gradients are close (gap 0.022), so the final stage is expensive and this fragmentation decides the row.

M3-R regroups exactly when the survivors stop sharing terms. Below, `regroup_active` is the number of survivors at the trigger (0 means it never fired), and the `parent_contexts` / `own_groups` columns show how scattered the two leading generators are across the parent groups.

In [13]:
diag = pd.DataFrame(dict(Case=summary.label, State=summary.state, M2=summary.M2.round(0).astype(np.int64),
                         M3=summary.M3.round(0).astype(np.int64), M3_R=summary.M3R.round(0).astype(np.int64),
                         M2_over_M3=(summary.M2 / summary.M3).round(2), M2_over_M3R=(summary.M2 / summary.M3R).round(2),
                         M3R_over_M3=(summary.M3R / summary.M3).round(3), regroup_active=summary.regroup_active,
                         new_contexts=summary.regroup_contexts,
                         winner_parent_contexts=summary.winner_parent_contexts, winner_own_groups=summary.winner_own_groups,
                         runner_parent_contexts=summary.runner_parent_contexts, runner_own_groups=summary.runner_own_groups))
diag.to_csv(f"{OUT}/table4b_M3_regroup.csv", index=False)
diag

,Case,State,M2,M3,M3_R,M2_over_M3,M2_over_M3R,M3R_over_M3,regroup_active,new_contexts,winner_parent_contexts,winner_own_groups,runner_parent_contexts,runner_own_groups
0,"LiH, R=3.0 A",HF,908972,416237,416237,2.18,2.18,1.000,0,0,444,52,449,51
1,"H4 square, side=1.0 A",HF,964052,355668,342963,2.71,2.81,0.964,2,9,14,8,15,8
2,"H4 square, side=1.0 A",CISD,102363,44972,44972,2.28,2.28,1.000,0,0,14,7,15,8
3,"H4 square, side=2.0 A",HF,456645,141420,141420,3.23,3.23,1.000,0,0,17,8,16,8
4,"H4 square, side=2.0 A",CISD,90471,29412,29412,3.08,3.08,1.000,0,0,10,7,15,8
5,"H2O eq., R(OH)=0.9572 A",HF,19510231,14034153,13965191,1.39,1.40,0.995,2,140,706,65,1000,78
6,"H2O eq., R(OH)=0.9572 A",CISD,35812518352,25208640396,25168979037,1.42,1.42,0.998,2,153,706,65,1022,74
7,"H2O stretch, R(OH)=1.75 A",HF,10719210,11421199,8449032,0.94,1.27,0.740,2,150,680,58,959,66
8,"H2O stretch, R(OH)=1.75 A",CISD,108186576,84904189,81839036,1.27,1.32,0.964,2,150,680,58,959,66


**Reading the regrouping table.**
* **The rule fires in five rows**: H₄ 1.0 HF and all four H₂O rows. In each, the last two survivors share no term. In the other four rows the final survivors still share terms (on LiH the runner-up is an exact spin-partner pair), so the parent groups keep a sharing benefit and nothing changes.
* **Regrouping never increases the cost**, because the parent shots are credited.
* **The saving is largest on H₂O stretch HF** (M3-R/M3 = 0.74), which turns $M2/M3=0.94$ into $M2/M3	ext{-R}=1.27$. After the fix, $M2/M3>1$ holds in all nine rows.
* **The saving depends on the final stage's share of the cost.** On H₂O stretch HF the final stage is 42% of M3, so shrinking its groups matters. On H₂O eq. HF and CISD the winner is just as scattered (706 parent contexts against 65 own groups), but the final stage is only 3–6% of M3, and the saving is below 1%.

### Table 5 — H₂O status
All four H₂O rows now have M1, M2 and M3, so the three-method comparison is complete for every Part I system.

In [14]:
h2o = summary.case.str.startswith("H2O")
t4[h2o.values]

,Case,State,M1_FCUG,M2_BAIFCIG,M3_BAIFCUG,M1_over_M2,M2_over_M3,M1_over_M3
5,"H2O eq., R(OH)=0.9572 A",HF,106461625,19510231,14034153,5.46,1.39,7.59
6,"H2O eq., R(OH)=0.9572 A",CISD,250962481286,35812518352,25208640396,7.01,1.42,9.96
7,"H2O stretch, R(OH)=1.75 A",HF,2190235853,10719210,11421199,204.33,0.94,191.77
8,"H2O stretch, R(OH)=1.75 A",CISD,3001532061,108186576,84904189,27.74,1.27,35.35


### Where M3 saves over M1
M3 needs the tightest radius only for the arms that are close to the winner. The stage log shows how the cost builds up as the active set shrinks.

In [15]:
stage_rows = []
for c, s in CASES:
    st = json.load(open(f"{OUT}/{c}_{s}_M3_BAIFCUG_stages.json"))
    for k, x in enumerate(st):
        stage_rows.append(dict(case=c, state=s, stage=k, radius=x["radius"], active=x["active"], stage_cost=x["stage_cost"]))
stages = pd.DataFrame(stage_rows)
stages.groupby(["case", "state"]).agg(stages=("stage", "count"), final_active=("active", "last"),
                                      final_radius=("radius", "last"), largest_stage_cost=("stage_cost", "max"))

stages  final_active  final_radius  largest_stage_cost
case     state                                                        
H2O_1.75 CISD       53             2      0.008478        7.555739e+07
         HF         33             2      0.010928        6.931911e+06
H2O_eq   CISD       53             2      0.001032        2.229698e+10
         HF         32             2      0.050545        1.170227e+07
H4_1.0   CISD        7             3      0.059041        3.658423e+04
         HF          7             2      0.010770        3.340330e+05
H4_2.0   CISD        7             2      0.025930        2.419782e+04
         HF          7             3      0.008324        1.391073e+05
LiH_3.0  HF         18             3      0.048194        3.704785e+05

## 9. Conclusions from the data

The printed tables above are the result; the points below summarize them.

1. **Gradients reproduce Part I.** The top generator, $|g_1|$ and $|g_2|$ agree with the Part I report in every row to $10^{-6}$ or better.
2. **M3 is the cheapest method in eight of nine rows; with regrouping, in all nine.** Plain M3 loses to M2 by 6% on H₂O at 1.75 Å (HF). The two final survivors share no Pauli term, and the winner is scattered over 680 parent contexts instead of its own 58 groups. The oracle-free regrouping rule (M3-R) fires in five rows, never increases the cost, and lowers this row by 26%. That gives M2/M3-R = 1.27, and M2/M3-R lies between 1.27 and 3.23 across all nine rows.
3. **M2 beats M1 in seven of nine rows.** The two exceptions are the H₄ CISD states, where the gap is large and the full-vector estimate at radius gap/2 is already cheap. Changing the target from full-vector estimation to winner identification pays off when the gap is small.
4. **Largest effect of BAI.** M1/M3 is largest on H₂O 1.75 Å HF (about 190), where a small gap forces M1 to resolve all 140 gradients very finely.
5. **Differences from the Part I preliminary numbers.** The H₂O M2 values are 17–19% above Part I. The H₄ M1 values are about 100 times lower, because our parent grouping packs the 1,276 Pauli products into 33–37 groups instead of 356. Our H₄ M2 values are 1.9–4.7 times higher and our LiH M2 is 8 times lower. The M3 values differ in both directions (0.33× for LiH, 4.7–14× for H₄). Part I did not record its grouping convention or its exact M2/M3 accounting, so these differences cannot be traced further. The definitions used here are written out in Section 5, and every number is backed by saved fragment variances.

The H₂O row of the Part I work order ("complete M3 for H₂O before making a headline all-system conclusion") is now closed.